In [11]:
import pandas as pd
import numpy as np
import re
import nltk
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from tabulate import tabulate
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


def clean_text(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'[^a-zA-Z\s]', '', text.lower())
    tokens = text.split()
    return " ".join([lemmatizer.lemmatize(w) for w in tokens if w not in stop_words])

# 2. Load Data
df = pd.read_csv("customer_complaints_1.csv")

column_name = 'text' 

# 3. Preprocess the specific column
df['cleaned_text'] = df[column_name].apply(clean_text)

# 4. Vectorize the content
vectorizer = TfidfVectorizer(max_features=1000) # Limit features for better clustering
X = vectorizer.fit_transform(df['cleaned_text'])

# 5. K-Means
k = 3 
km = KMeans(n_clusters=k, random_state=42)
df['cluster'] = km.fit_predict(X)

# 6. Display Results (Showing first 10 rows)
table_data = [["Original Text Snippet", "Cluster"]]
for i, row in df.head(10).iterrows():
    # Truncate text for readability in the table
    snippet = (row[column_name][:50] + '..') if len(row[column_name]) > 50 else row[column_name]
    table_data.append([snippet, row['cluster']])

print(tabulate(table_data, headers="firstrow"))

# 7. Top Terms per Cluster
print("\nTop terms per cluster:")
order_centroids = km.cluster_centers_.argsort()[:, ::-1]
terms = vectorizer.get_feature_names_out()
for i in range(k):
    print(f"Cluster {i}: ", end="")
    print(", ".join([terms[ind] for ind in order_centroids[i, :8]]))

# Calculate purity
total_samples = len(y_pred)
cluster_label_counts = [Counter(y_pred)]
purity = sum(max(cluster.values()) for cluster in cluster_label_counts) / total_samples
print("Purity:", purity)


Original Text Snippet                                   Cluster
----------------------------------------------------  ---------
I used to love Comcast. Until all these constant u..          1
I'm so over Comcast! The worst internet provider. ..          1
If I could give them a negative star or no stars o..          2
I've had the worst experiences so far since instal..          0
Check your contract when you sign up for Comcast a..          2
Thank God. I am changing to Dish. They gave me awe..          1
I Have been a long time customer and only have Xfi..          1
There is a malfunction on the DVR manager which is..          0
Charges overwhelming. Comcast service rep was so i..          2
I have had cable, DISH, and U-verse, etc. in the p..          1

Top terms per cluster:
Cluster 0: second, box, adding, malfunction, protocol, investigating, since, floor
Cluster 1: internet, service, day, mbps, speed, xfinity, comcast, customer
Cluster 2: rude, comcast, service, contract, issue